In [1]:
# --- Notebook Test Engine: injected setup (auto-generated) ---
import os as _os
for _k in ('AWS_ACCESS_KEY_ID','AWS_SECRET_ACCESS_KEY','AWS_SESSION_TOKEN','AWS_CREDENTIAL_EXPIRATION'):
    _os.environ.pop(_k, None)
_os.environ.setdefault('AWS_DEFAULT_REGION', 'us-west-2')
_os.environ.setdefault('AWS_REGION', 'us-west-2')
_nte_role = 'arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole'
try:
    from sagemaker.core.helper import session_helper as _sh
    _sh.get_execution_role = lambda *a, **k: _nte_role
except Exception:
    pass
try:
    _nte_cf = _os.environ.get('AWS_SHARED_CREDENTIALS_FILE')
    if _nte_cf and _os.path.exists(_nte_cf):
        import configparser as _cfp, datetime as _dt
        import boto3 as _b3, botocore.session as _bcs
        from botocore.credentials import RefreshableCredentials as _RC
        _nte_prof = _os.environ.get('AWS_PROFILE', 'default')
        def _nte_load():
            _cp = _cfp.ConfigParser(); _cp.read(_nte_cf)
            _s = _cp[_nte_prof]
            _tok = _s.get('aws_session_token')
            if not _tok:
                raise RuntimeError('nte: no session token in creds file')
            return {'access_key': _s['aws_access_key_id'],
                    'secret_key': _s['aws_secret_access_key'],
                    'token': _tok,
                    'expiry_time': (_dt.datetime.now(_dt.timezone.utc)
                                    + _dt.timedelta(minutes=9)).isoformat()}
        _nte_creds = _RC.create_from_metadata(metadata=_nte_load(),
                                              refresh_using=_nte_load,
                                              method='nte-shared-file')
        def _nte_get_credentials(self, *a, **k):
            # Honor sessions that were handed explicit credentials
            # (e.g. notebooks that assume_role into other accounts):
            # clobbering them would silently run every cross-account
            # client as the ambient identity. Bare/SDK-created sessions
            # (no explicit creds) still get the refreshable shared-file
            # creds so long-running waits survive credential rotation.
            _ex = getattr(self, '_credentials', None)
            if _ex is not None:
                return _ex
            return _nte_creds
        _bcs.Session.get_credentials = _nte_get_credentials
        _b3.setup_default_session(botocore_session=_bcs.get_session())
        print('nte: refreshable shared-file creds installed')
except Exception as _e:
    print('nte: refreshable-creds shim skipped:', _e)
try:
    from sagemaker.core.resources import ModelPackageGroup as _NteMPG
    _nte_mpg_create = _NteMPG.create
    def _nte_mpg_get_or_create(*_a, **_k):
        try:
            return _nte_mpg_create(*_a, **_k)
        except Exception as _e2:
            if 'already exists' in str(_e2).lower():
                _name = _k.get('model_package_group_name') or (_a[0] if _a else None)
                return _NteMPG.get(_name)
            raise
    _NteMPG.create = staticmethod(_nte_mpg_get_or_create)
    print('nte: ModelPackageGroup.create is now get-or-create')
except Exception as _e:
    print('nte: ModelPackageGroup idempotency shim skipped:', _e)


nte: ModelPackageGroup.create is now get-or-create


# Fine-Tuning with Serverful Training Jobs (SMTJ)

This notebook demonstrates fine-tuning Amazon Nova models using `SFTTrainer` on
**serverful SageMaker Training Job** instances with **recipe overrides**.

## What you will learn

1. Create an `SFTTrainer` with `TrainingJobCompute`
2. Use YAML recipe files with selective overrides
3. Inspect the resolved recipe before submission
4. Submit and monitor a training job
5. Use the low-level `ModelTrainer.from_recipe()` API

## 1. Setup

In [2]:
import json
import boto3

In [3]:
# === Fill in your AWS resources ===
REGION = "us-west-2" # e.g. "us-east-1"
ROLE_ARN = "arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole"
S3_BUCKET = "notebook-test-engine-ds-674622101542-usw2" # e.g. "sagemaker-us-east-1-674622101542"

S3_OUTPUT_PATH = f"s3://{S3_BUCKET}/sft-smtj/output"
TRAINING_DATASET = "s3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-converse-messages.jsonl"

## 2. Create SFTTrainer

Create a trainer with serverful compute (`TrainingJobCompute`).
This provisions dedicated instances for the training job.

In [4]:
from sagemaker.train import SFTTrainer
from sagemaker.train.common import TrainingType
from sagemaker.core.training.configs import TrainingJobCompute

sft_trainer = SFTTrainer(accept_eula=True, 
    model="amazon.nova-2-lite-v1",
    training_type=TrainingType.LORA,
    training_dataset=TRAINING_DATASET,
    s3_output_path=S3_OUTPUT_PATH,
    compute=TrainingJobCompute(
        instance_type="ml.p5.48xlarge",
        instance_count=4,
    ),
    role=ROLE_ARN,
    base_job_name="sft-smtj",
)

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


[07/20/26 20:39:05] INFO     SageMaker session not provided. Using default Session.                  ]8;id=10478770;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=10478771;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     Runs on sagemaker prod, region:us-west-2                                  ]8;id=10478778;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py\utils.py]8;;\:]8;id=10478779;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/utils/utils.py#375\375]8;;\

## 3. Using Recipe Overrides

You can provide a YAML recipe file with your training configuration, then selectively
override specific parameters via the `overrides` dict. The override takes precedence
over values in the recipe file.

This is useful when you want a shared base recipe but need to experiment with specific
hyperparameters (e.g., learning rate or number of epochs) without modifying the file.

**Precedence order:** `overrides dict` > `recipe YAML` > `Hub/SDK defaults`

In [5]:
import yaml

# Write a custom recipe YAML
recipe_config = {
    "training": {
        "learning_rate": 1e-5,
        "num_epochs": 3,
        "batch_size": 8,
        "sequence_length": 2048,
    }
}

with open("my_sft_recipe.yaml", "w") as f:
    yaml.dump(recipe_config, f)

print("Recipe file contents:")
print(yaml.dump(recipe_config, default_flow_style=False))

Recipe file contents:
training:
  batch_size: 8
  learning_rate: 1.0e-05
  num_epochs: 3
  sequence_length: 2048



In [6]:
# Create trainer with recipe + overrides
# Here we override learning_rate (1e-5 -> 5e-6) and num_epochs (3 -> 5)
# from the recipe file above, while keeping batch_size and sequence_length unchanged
sft_trainer_with_recipe = SFTTrainer(accept_eula=True, 
    model="amazon.nova-2-lite-v1",
    training_type=TrainingType.LORA,
    training_dataset=TRAINING_DATASET,
    s3_output_path=S3_OUTPUT_PATH,
    compute=TrainingJobCompute(
        instance_type="ml.p5.48xlarge",
        instance_count=4,
    ),
    role=ROLE_ARN,
    recipe="my_sft_recipe.yaml",
    overrides={"training_config": {"learning_rate": 5e-6, "num_epochs": 5}},
    base_job_name="sft-recipe-override-smtj",
)

# Inspect the resolved recipe to confirm overrides were applied
resolved = sft_trainer_with_recipe.get_resolved_recipe()
print("Resolved training_config:")
print(json.dumps(resolved.get("training_config", resolved), indent=2))

[07/20/26 20:39:06] INFO     SageMaker session not provided. Using default Session.                  ]8;id=10478784;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=10478785;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     SageMaker session not provided. Using default Session.                  ]8;id=10478790;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=10478791;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/20/26 20:39:07] WARNING  Override key 'training' from user recipe (my_sft_recipe.yaml)   ]8;id=10478798;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/recipe_resolver.py\recipe_resolver.py]8;;\:]8;id=10478799;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/recipe_resolver.py#549\549]8;;\
                             does not exist in the recipe and will be dropped.                                     

                    WARNING  Override key 'training_config.num_epochs' from overrides does   ]8;id=10478804;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/recipe_resolver.py\recipe_resolver.py]8;;\:]8;id=10478805;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/recipe_resolver.py#549\549]8;;\
                             not exist in the recipe and will be dropped.                                          

Resolved training_config:
{
  "max_steps": 10,
  "save_steps": 10,
  "save_top_k": 5,
  "max_length": 32768,
  "global_batch_size": 32,
  "reasoning_enabled": true,
  "val_check_interval": 2500,
  "limit_val_batches": 2,
  "lr_scheduler": {
    "warmup_steps": 15,
    "min_lr": 1e-06
  },
  "optim_config": {
    "lr": 5e-06,
    "weight_decay": 0.0,
    "adam_beta1": 0.9,
    "adam_beta2": 0.95
  },
  "peft": {
    "peft_scheme": "lora",
    "lora_tuning": {
      "alpha": 64,
      "lora_plus_lr_ratio": 64.0
    }
  },
  "model_importance_score": {
    "fine_tuned_model": 1.0
  }
}


## 4. Set Hyperparameters and Submit Training Job

In [7]:
sft_trainer.hyperparameters.max_steps = 50
sft_trainer.hyperparameters.learning_rate = 5e-6
sft_trainer.hyperparameters.global_batch_size = 32

# Submit (non-blocking)
training_job = sft_trainer.train(wait=False)
print(f"Training job submitted: {training_job}")

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=10478812;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=10478813;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

[07/20/26 20:39:08] INFO     SageMaker session not provided. Using default Session.                  ]8;id=10478818;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=10478819;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

[07/20/26 20:39:09] INFO     Cannot simulate policies for                                  ]8;id=10478826;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=10478827;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=10478833;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=10478834;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     SMTJ recipe S3 URI:                                                ]8;id=10478841;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/base_trainer.py\base_trainer.py]8;;\:]8;id=10478842;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/base_trainer.py#413\413]8;;\
                             s3://jumpstart-cache-prod-us-west-2/recipes/nova_lite_2_0_p5_gpu_l                    
                             ora_sft_payload_template_sm_jobs_v2.6.0.yaml                                          

...

                    INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=10478906;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=10478907;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/telemetry/telemetry_logging.py#288\288]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml


sagemaker.config INFO - Fetched defaults config from location: /opt/ml/processing/input/sm_config.yaml


[07/20/26 20:39:13] INFO     Creating training_job resource.                                     ]8;id=10478914;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py\resources.py]8;;\:]8;id=10478915;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/resources.py#31239\31239]8;;\

Training job submitted: training_job_name='sft-smtj-20260720203912' training_job_arn='arn:aws:sagemaker:us-west-2:674622101542:training-job/sft-smtj-20260720203912' tuning_job_arn=Unassigned() labeling_job_arn=Unassigned() auto_ml_job_arn=Unassigned() model_artifacts=Unassigned() training_job_status='InProgress' secondary_status='Starting' failure_reason=Unassigned() hyper_parameters={'adam_beta1': '0.9', 'adam_beta2': '0.95', 'alpha': '64', 'base_model': 'nova-lite-2/prod', 'checkpoint_s3_uri': 's3://customer-escrow-674622101542-smtj-a3e2c21d/sft-smtj-20260720203912', 'data_s3_path': 's3://notebook-test-engine-ds-674622101542-usw2/datasets/sample-converse-messages.jsonl', 'fine_tuned_model': '1.0', 'global_batch_size': '32', 'learning_rate': '5e-06', 'learning_rate_ratio': '64.0', 'limit_val_batches': '2', 'lora_alpha': '64', 'lora_plus_lr_ratio': '64.0', 'lr': '5e-06', 'max_context_length': '8192', 'max_length': '32768', 'max_steps': '50', 'min_lr': '1e-06', 'mlflow_experiment_name':

## 5. Monitor Training Job

In [8]:
from sagemaker.core.resources import TrainingJob

job = TrainingJob.get(training_job_name=training_job.training_job_name)
print(f"Status: {job.training_job_status}")
print(f"Secondary Status: {job.secondary_status}")

Status: InProgress
Secondary Status: Starting


## 6. Low-Level Alternative: ModelTrainer.from_recipe

For maximum control, use `ModelTrainer.from_recipe()` directly.
This gives you full access to the recipe structure without the high-level trainer abstraction.

In [9]:
import yaml
from sagemaker.train import ModelTrainer
from sagemaker.train.configs import Compute

yaml.dump({
    "run": {
        "model_type": "amazon.nova.lite",
        "model_name_or_path": "nova-textgeneration-lite-v2",
        "replicas": 1,
    },
    "training_config": {
        "learning_rate": 1e-5,
        "num_epochs": 3,
        "batch_size": 4,
    },
}, open("my_nova_recipe.yaml", "w"))

model_trainer = ModelTrainer.from_recipe(
    training_recipe="my_nova_recipe.yaml",
    compute=Compute(instance_type="ml.p5.48xlarge", instance_count=1),
    training_image="<your-nova-training-image-uri>",
    recipe_overrides={"training_config": {"learning_rate": 5e-6, "num_epochs": 5}},
)

resolved = model_trainer.get_resolved_recipe()
print(json.dumps(resolved.get("training_config", resolved), indent=2))

[07/20/26 20:39:15] INFO     SageMaker session not provided. Using default Session.                  ]8;id=10478920;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=10478921;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#65\65]8;;\

                    INFO     Cannot simulate policies for                                  ]8;id=10478926;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=10478927;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=10478932;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=10478933;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#585\585]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Role not provided. Using validated caller role:                         ]8;id=10478939;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=10478940;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#90\90]8;;\
                             arn:aws:iam::674622101542:role/NotebookTestEngine-ProcessingJobRole                   

[07/20/26 20:39:16] INFO     Cannot simulate policies for                                  ]8;id=10478945;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=10478946;file:///usr/local/lib/python3.12/dist-packages/sagemaker/core/helper/iam_role_resolver.py#363\363]8;;\
                             'arn:aws:iam::674622101542:role/NotebookTestEngine-Processing                         
                             JobRole' (access denied); permission verdict unknown.                                 

...

                    INFO     Base name not provided. Using default name:                            ]8;id=10478958;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=10478959;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#129\129]8;;\
                             <your-nova-training-image-uri>-job                                                    

                    INFO     StoppingCondition not provided. Using default:                         ]8;id=10478964;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=10478965;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#167\167]8;;\
                             max_runtime_in_seconds=3600 max_wait_time_in_seconds=None                             
                             max_pending_time_in_seconds=None                                                      

                    INFO     OutputDataConfig not provided. Using default:                          ]8;id=10478971;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=10478972;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/defaults.py#192\192]8;;\
                             s3_output_path='s3://sagemaker-us-west-2-674622101542/<your-nova-train                
                             ing-image-uri>-job' kms_key_id=None compression_type='GZIP'                           

                    INFO     Training image URI: <your-nova-training-image-uri>                ]8;id=10478977;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=10478978;file:///usr/local/lib/python3.12/dist-packages/sagemaker/train/model_trainer.py#558\558]8;;\

{
  "batch_size": 4,
  "learning_rate": 5e-06,
  "num_epochs": 5
}
